<a href="https://colab.research.google.com/github/mario-rutman/02_data_tyding_project/blob/master/Sin%C3%B4nimos_e_express%C3%B5es_equivalentes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U google-generativeai

## Encontrando Sinônimos e Expressões Equivalentes em Português com a API Gemini

### Sempre que for usar o API rodar
#### Importe o SDK Python
#### Inicialize o modelo Gemini atualizado
#### crie a função encontrar_sinonimos_e_expressoes_df(palavra)
#### exemplo de uso.

In [ ]:
# Importe o SDK Python
import google.generativeai as genai
# Usado para armazenar de forma segura sua chave de API
from google.colab import userdata

GOOGLE_API_KEY=userdata.get('chave_01')
genai.configure(api_key=GOOGLE_API_KEY)

print("Chave de API configurada com sucesso!")

Agora, vamos inicializar o modelo Generativo. Escolhi `gemini-pro` por sua capacidade de compreender e gerar texto complexo.

In [ ]:
# Inicialize o modelo Gemini atualizado
gemini_model = genai.GenerativeModel('models/gemini-flash-lite-latest')
print("Modelo Gemini inicializado!")

### Função para Buscar Sinônimos e Expressões

A função `encontrar_sinonimos_e_expressoes` usará o modelo Gemini para gerar uma lista de sinônimos e expressões equivalentes para a palavra fornecida. O prompt é cuidadosamente construído para pedir um formato de saída claro e útil para sua escrita.

In [ ]:
import json
import pandas as pd


def encontrar_sinonimos_e_expressoes_df(palavra):
  """Encontra sinônimos e expressões equivalentes e retorna um DataFrame pandas."""
  prompt = f"""Para a palavra em português '{palavra}', forneça:
    1. Uma lista de 5 a 10 sinônimos.
    2. Uma lista de 5 a 10 expressões equivalentes.

    Retorne APENAS um JSON válido no seguinte formato, sem formatação Markdown adicional:
    {{
      "sinonimos": ["sinônimo 1", "sinônimo 2", ...],
      "express_equival": ["expressão 1", "expressão 2", ...],
      "antonimos":["antônimo 1", "antônimo 2", ...]
      }}"""

  try:
    # Garante que a resposta venha no formato JSON
    response = gemini_model.generate_content(
        prompt,
        generation_config=genai.types.GenerationConfig(
            response_mime_type="application/json"
        ),
    )

    dados = json.loads(response.text)

    # Prepara listas do mesmo tamanho para criar o DataFrame
    sinonimos = dados.get("sinonimos", [])
    expressoes = dados.get("express_equival", [])
    antonimos = dados.get("antonimos", [])

    # Ajusta os tamanhos para alinhar nas colunas da tabela
    max_len = max(len(sinonimos), len(expressoes), len(antonimos))
    sinonimos += [None] * (max_len - len(sinonimos))
    expressoes += [None] * (max_len - len(expressoes))
    antonimos += [None] * (max_len - len(antonimos))

    # Cria o DataFrame com o nome exato das colunas solicitadas
    df = pd.DataFrame({"sinonimo": sinonimos, "express_equival": expressoes, "antonimo": antonimos})

    return df

  except Exception as e:
    print(f"Ocorreu um erro: {e}")
    return None

### Exemplo de Uso

In [ ]:
df_result = encontrar_sinonimos_e_expressoes_df('sono')
df_result